# Post-Training Quantisation (PTQ)

Purpose: converts the FP32 baseline to a full int8 TFLite model using post-training quantisation. There is no retraining happening in this notebook

In [ ]:
# import
import tensorflow as tf
import numpy as np
import os
import time
import csv

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/tinyml-quant-security'

In [ ]:
# Load CIFAR-10 and use the same fixed split as notebook
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

split = np.load(f'{PROJECT_DIR}/results/data_split.npz')
train_idx, val_idx = split['train_idx'], split['val_idx']

x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

In [ ]:
# Load the trained FP32 baseline from the notebook 'baseline_training.py'
saved_model_path = f'{PROJECT_DIR}/models/baseline_savedmodel'

In [ ]:
""" Why calibration?
Ans: PTQ needs the scale factor because it can represent −3.4×10^38 to +3.4×10^38
in Float3, but only -128 to 127 in Int8. So, in this cell, it will use some images
to calibrate in every layer
"""
NUM_CALIBRATION_SAMPLES = 300 # this 300 images are randomly drawn from the training set

rng = np.random.RandomState(SEED)
calib_indices = rng.choice(len(x_train), NUM_CALIBRATION_SAMPLES, replace=False)
calib_data = x_train[calib_indices]

def representative_dataset():
    for i in range(len(calib_data)):
        sample = calib_data[i:i+1].astype(np.float32)
        yield [sample]

In [ ]:
# Convert to full int8 TFLite model
"""
1. Runs all 300 calibration images through the model
2. Calculates scale and zero point for every layer
3. Converts all float32 weights to int8
4. Converts all activation ranges to int8
"""
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset # the calibration will be used here

# make both input and output to int8
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

ptq_tflite_model = converter.convert()

ptq_model_path = f'{PROJECT_DIR}/models/ptq_model.tflite'
with open(ptq_model_path, 'wb') as f:
    f.write(ptq_tflite_model)

print(f"Saved PTQ model to {ptq_model_path}")
print(f"PTQ model size: {os.path.getsize(ptq_model_path) / 1024:.2f} KB")

## Clean evaluation

Since the PTQ model uses int8 input/output, test images must be quantised to int8 using the same scale the converter computed, before being passed into the interpreter.

In [ ]:
# Loading TFLite interpreter
interpreter = tf.lite.Interpreter(model_path=ptq_model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

input_scale, input_zero_point = input_details['quantization']
print(f"Input quantisation - scale: {input_scale}, zero_point: {input_zero_point}")

In [ ]:
def quantize_input(x_float, scale, zero_point):
    x_int8 = x_float / scale + zero_point
    return np.clip(np.round(x_int8), -128, 127).astype(np.int8)

def evaluate_tflite_model(interpreter, x_test, y_test, input_scale, input_zero_point, num_samples=None):
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    if num_samples is None:
        num_samples = len(x_test)

    correct = 0
    latencies = []

    for i in range(num_samples):
        x_sample = quantize_input(x_test[i:i+1], input_scale, input_zero_point)

        start = time.perf_counter()
        interpreter.set_tensor(input_details['index'], x_sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])
        latencies.append(time.perf_counter() - start)

        pred = np.argmax(output[0])
        if pred == y_test[i]:
            correct += 1

    accuracy = correct / num_samples
    avg_latency_ms = np.mean(latencies) * 1000
    p95_latency_ms = np.percentile(latencies, 95) * 1000

    return accuracy, avg_latency_ms, p95_latency_ms

In [ ]:
# Evaluate on the full test set (10,000 samples)
# Reduce num_samples for a faster check
ptq_accuracy, ptq_avg_latency, ptq_p95_latency = evaluate_tflite_model(
    interpreter, x_test, y_test, input_scale, input_zero_point
)

ptq_size_kb = os.path.getsize(ptq_model_path) / 1024

print(f"PTQ - Accuracy: {ptq_accuracy:.4f}")
print(f"PTQ - Avg latency: {ptq_avg_latency:.3f} ms, P95 latency: {ptq_p95_latency:.3f} ms")
print(f"PTQ - Model size: {ptq_size_kb:.2f} KB")

In [ ]:
# Append results to shared CSV log (will be used later to compare all other variants)
results_path = f'{PROJECT_DIR}/results/clean_eval.csv'
file_exists = os.path.isfile(results_path)

with open(results_path, 'a', newline='') as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(['variant', 'accuracy', 'avg_latency_ms', 'p95_latency_ms', 'size_kb'])
    writer.writerow(['PTQ', ptq_accuracy, ptq_avg_latency, ptq_p95_latency, ptq_size_kb])

print(f"Appended PTQ results to {results_path}")